# coen-dome-decision-making: performance, water and probes

Reads this workflow's own sessions (`sub-<mouse>/ses-<id>_<DateTime>`, as the lab's ingest puts them on jacuzzi, Live-Data) with the columns each session's manifest carries (`schemas/trial-columns.yml`). Sessions of other workflows in the same root are skipped by their manifest. Performance is Correct over Correct plus Incorrect on new draws: outcome `Probe` trials (target `NoReward` or `Random`) and `NoResponse` are left out.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from modules.loading_functions import load_sessions, probe_rates, session_summary, trial_weights

In [ ]:
# Where the sessions are: jacuzzi's Live-Data (J: is jacuzzi's root on Windows). Set DOME_DECISION_ROOT or edit this line;
# never a production av-dome folder.
ROOT = Path(os.environ.get("DOME_DECISION_ROOT", r"J:\data\RDSS\Live-Data"))
SUBJECTS = None   # None: every mouse, or a list of lab IDs, e.g. ["GN013", "GN014"]

In [ ]:
df = load_sessions(ROOT, SUBJECTS)
summary = session_summary(df) if len(df) else pd.DataFrame()
summary

Learning curves: performance per session, one panel per mouse (scored trials only).

In [ ]:
mice = sorted(summary.subject.unique()) if len(summary) else []
fig, axes = plt.subplots(1, max(1, len(mice)), figsize=(3.5 * max(1, len(mice)), 3), squeeze=False)
for ax, mouse in zip(axes[0], mice):
    s = summary[summary.subject == mouse]
    ax.axhline(0.5, color="r", ls="--")
    ax.plot(range(1, len(s) + 1), s.performance, "-o")
    ax.set(title=mouse, xlabel="Session", ylabel="Performance", ylim=(0, 1))
plt.tight_layout()
plt.show()

Water per session: the sum of `rewardOfferedUl` over `rewarded` trials.

In [ ]:
summary[["subject", "session", "waterUl"]] if len(summary) else summary

Probe trials (`outcome` Probe, `target` NoReward or Random): how often they were rewarded and penalised.

In [ ]:
probe_rates(df) if len(df) else pd.DataFrame()

The trial weights a session ran with: the stage's own, else the top-level `trialWeights` (stage = the session's TrainingStage row).

In [ ]:
# trial_weights(<session folder>, stage=<TrainingStage>); e.g. the last session of the first mouse:
# trial_weights(df.session_path.iloc[-1], stage=0)